# Binning study: does a finer reco binning improve the measurement?

`xml/binning_study/run_all.sh` fits each XML family (`nuwro` fake data, `asimov` GENIE fake
data, optionally with a non-default chi^2 statistic such as `nuwro_cnp`) with several reco
log10(Q^2) x p_n binnings (variants), writing `<STUDY_ROOT>/<family>/<variant>/<fit>/`. This
notebook compares the variants of one family fit by fit, using only binning-independent
quantities, and ends with a cross-family overlay:

| figure of merit | source | reads as |
|---|---|---|
| $\chi^2_{\rm data}/N_{\rm dof}$ | `profile.log`, as in notebook 3 | goodness of fit; only comparable per degree of freedom |
| posterior width of the fitted parameters | MCMC chain via `load_fit` | narrower = more constraining |
| $\delta_F(Q^2)=\sigma[F_A]/|\mathbb{E}[F_A]|$ at $Q^2_{\rm ref}$ and versus $Q^2$ | MCMC chain, definition of notebook 6 | the headline precision; smaller = better |
| shift of the best fit relative to the reference variant | chain medians | a large shift on fake data means the finer bins resolve model differences, not just add statistics |
| profile scans $\Delta\chi^2(\theta)$ | `*_PROfile_points.txt` | shape of the likelihood per parameter |
| axial radius $r_A^2$ and the fraction of samples with $r_A^2<0$ | MCMC chain, definitions of notebook 7 | the slope of $F_A$ at $Q^2=0$; unphysical negative values signal an unconstrained slope |

Variants that have not finished yet are skipped and listed in the inventory, so the notebook
can be rerun while the study is still running. `nominal` is the production binning and is the
reference every ratio is taken against.

The Asimov family is the control: its fake data *is* the GENIE prediction, so any drift of the
best fit with the binning there is an estimator artifact (Neyman bias, spline range), not
physics. `nuwro_pm3` holds the first batch, run with the production +-3 spline fit box; the
current XMLs widen the axial splines to +-4.

In [ ]:
from pathlib import Path
import os
import re
import sys

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

# Locate ma_zexp whether the notebook starts here or from axial_mass.
start = Path.cwd().resolve()
REPO = next(
    (parent / 'ma_zexp' for parent in (start, *start.parents)
     if (parent / 'ma_zexp' / 'python' / 'scripts' / 'postfit_physical_parameters.py').is_file()),
    None,
)
if REPO is None:
    raise FileNotFoundError('Could not locate ma_zexp/python/scripts/postfit_physical_parameters.py')
if str(REPO / 'python' / 'scripts') not in sys.path:
    sys.path.insert(0, str(REPO / 'python' / 'scripts'))

from postfit_physical_parameters import PUBLICATION_RC, SPECS, _fa_curves, load_fit

mpl.rcParams.update(PUBLICATION_RC)
SPEC = {spec.key: spec for spec in SPECS}

## Controls

`STUDY_ROOT` is where `binning_study/run_all.sh` wrote its results and `FAMILY` selects the XML
family to analyse (environment overrides: `BINNING_STUDY_ROOT`, `BINNING_STUDY_FAMILY`). The
family directory name may carry a suffix such as `_cnp` or `_pm3`; the XML edges are read from
`xml/binning_study/<family without suffix>/`. Leave `FIT_KEYS = None` to compare every fit that
has at least one finished variant, or list the keys you care about.

`RESOLUTION_SAFE_ONLY`, off by default, restricts every section below to variants whose bins are
at least `RESOLUTION_FACTOR` times the local reco resolution wide on both axes (the same guard
notebook 17 grows its count-based grids under). This drops any variant, including `nominal` if it
fails, that leans on migration between neighbouring bins rather than independent information; if
`REFERENCE_VARIANT` itself is not resolution-safe the cell raises rather than silently comparing
against an unsafe reference.

In [ ]:
STUDY_ROOT = Path(os.environ.get('BINNING_STUDY_ROOT', '/nevis/hopper/data/epelaez/axial_mass/binning_study_fit_results'))
XML_ROOT = REPO / 'xml' / 'binning_study'
FAMILY = os.environ.get('BINNING_STUDY_FAMILY', 'nuwro')   # nuwro, asimov, nuwro_cnp, nuwro_pm3, ...
# Families overlaid in the last section (only those that exist are used).
COMPARE_FAMILIES = ('nuwro_pm3', 'nuwro', 'nuwro_cnp', 'asimov')
COMPARE_FITS = ('minerva_k6', 'minerva_k6_uniform', 'ma', 'lqcd_k6')
REFERENCE_VARIANT = 'count100'
FIT_KEYS = None            # e.g. ('minerva_k6', 'lqcd_k6', 'minerva_k6_uniform')
Q2_REFERENCE = 0.50        # GeV^2, the comparison point of notebook 6
Q2 = np.linspace(0.0, 2.0, 201)
BURN_IN, THIN = 0, 1
Q2_CHUNK = 25              # Q^2 points evaluated at once: every chain sample is used, chunking only bounds memory
PROFILE_SCAN_FIT = 'ma_no_axff'   # fit whose 1D profile scans are overlaid in the last section

# Notebook 17 rejects any candidate bin narrower than RESOLUTION_FACTOR times the local reco
# resolution on either axis; turning this on applies the same guard here and drops every variant
# (nominal's hand-drawn grid included) that violates it on at least one bin, so the comparison is
# restricted to binnings where migration between neighbouring bins is not the dominant effect.
RESOLUTION_SAFE_ONLY = True
RESOLUTION_FACTOR = 1.0
RESOLUTION_INPUT_FILE = '/nevis/riverside/data/epelaez/ngem/intermediate_files/minimal_withspline_df.root'

FIT_ORDER = [
    'ma', 'ma_no_axff', 'ma_uniform', 'minerva_k8', 'minerva_k7',
    'minerva_k6', 'lqcd_k7', 'lqcd_k6', 'minerva_lqcd_k7', 'minerva_lqcd_k6',
    'minerva_k6_nuisance', 'lqcd_k6_nuisance', 'minerva_lqcd_k6_nuisance',
    'minerva_k6_uniform', 'minerva_k6_uniform_nuisance',
]
# Fitted-parameter counts and uniform (penalty-free) parameters, as in notebook 3.
N_FITTED = {
    'ma': 4, 'ma_uniform': 3, 'ma_no_axff': 3,
    'lqcd_k6': 2, 'lqcd_k7': 3, 'minerva_k6': 2, 'minerva_k7': 3,
    'minerva_k8': 4, 'minerva_lqcd_k6': 2, 'minerva_lqcd_k7': 3,
    'minerva_k6_uniform': 2, 'minerva_k6_nuisance': 4, 'lqcd_k6_nuisance': 4,
    'minerva_lqcd_k6_nuisance': 4, 'minerva_k6_uniform_nuisance': 4,
}
UNIFORM_PARAMETERS = {
    'ma_uniform': {'MACCQE'},
    'minerva_k6_uniform': {'FAzexpMinervaK6PCA1', 'FAzexpMinervaK6PCA2'},
    'minerva_k6_uniform_nuisance': {'FAzexpMinervaK6PCA1', 'FAzexpMinervaK6PCA2'},
}

## 1. Inventory

Variants are the subdirectories of `STUDY_ROOT`, ordered by their number of reco bins (read
from the XMLs). A fit counts as finished when its `*_v1_PROfile.root` exists.

In [ ]:
RECO_UNIT = 'log10(Q^2 / GeV^2);p_n'


def xml_family(family):
    '''XML directory of a results family: strip result-only suffixes such as _cnp or _pm3.'''
    if (XML_ROOT / family).is_dir():
        return family
    base = family.split('_')[0]
    if (XML_ROOT / base).is_dir():
        return base
    raise FileNotFoundError(f'no XML family for {family!r} below {XML_ROOT}')


def reco_edges(variant, family=None):
    '''(log10 Q^2 edges, p_n edges) from the first XML of the variant, or None.'''
    xmls = sorted((XML_ROOT / xml_family(family or FAMILY) / variant).glob('*.xml'))
    if not xmls:
        return None
    text = xmls[0].read_text()
    for block in re.findall(r'<bins2D\b.*?/>', text, flags=re.DOTALL):
        unit = re.search(r'unit="([^"]*)"', block)
        if unit and unit.group(1) == RECO_UNIT:
            return tuple(
                tuple(float(v) for v in re.search(f'{attr}="([^"]*)"', block).group(1).split())
                for attr in ('edgesx', 'edgesy')
            )
    return None


def profile_root(variant, fit, family=None):
    matches = sorted((STUDY_ROOT / (family or FAMILY) / variant / fit).glob('*_v1_PROfile.root'))
    return matches[0] if len(matches) == 1 else None


def scan_family(family):
    '''variant -> dict(nbins, edges, finished fits) for one results family.'''
    root = STUDY_ROOT / family
    if not root.is_dir():
        raise FileNotFoundError(f'{root} does not exist; run xml/binning_study/run_all.sh --family {family.split("_")[0]}')
    info = {}
    for directory in sorted(p for p in root.iterdir() if p.is_dir()):
        edges = reco_edges(directory.name, family)
        nbins = (len(edges[0]) - 1) * (len(edges[1]) - 1) if edges else np.nan
        finished = [fit for fit in FIT_ORDER if profile_root(directory.name, fit, family) is not None]
        info[directory.name] = dict(nbins=nbins, edges=edges, finished=finished)
    return info


def reco_resolution(input_file):
    '''(resolution_at_q2, resolution_at_pn): reco resolution vs reco value, as in notebook 17.

    Half the 16-84% spread of true minus reco, in slices of the reco variable, from signal
    overlay events; held constant outside the slice range covered by the sample.
    '''
    import uproot

    arrays = uproot.open(input_file)['tree'].arrays(
        ['isdata', 'isext', 'isdirt', 'isnuwro', 'afro_1mu1p_sel', 'afro_1mu1p_true',
         'afro_1mu1p_Q2', 'afro_1mu1p_Pn', 'afro_1mu1p_true_Pn', 'GTruth_gQ2'],
        library='np',
    )
    overlay = ((arrays['isdata'] == 0) & (arrays['isext'] == 0) & (arrays['isdirt'] == 0)
               & (arrays['isnuwro'] == 0) & (arrays['afro_1mu1p_sel'] == 1))
    signal = overlay & (arrays['afro_1mu1p_true'] == 1)
    with np.errstate(divide='ignore', invalid='ignore'):
        logq2, logq2_true = np.log10(arrays['afro_1mu1p_Q2']), np.log10(arrays['GTruth_gQ2'])
    pn, pn_true = arrays['afro_1mu1p_Pn'], arrays['afro_1mu1p_true_Pn']

    def slice_resolution(reco, true, edges, min_events=50):
        residual = true - reco
        centers, sigmas = [], []
        for lo, hi in zip(edges[:-1], edges[1:]):
            part = residual[(reco >= lo) & (reco < hi)]
            if len(part) < min_events:
                continue
            q16, q84 = np.percentile(part, [16, 84])
            centers.append(0.5 * (lo + hi)); sigmas.append(0.5 * (q84 - q16))
        return np.asarray(centers), np.asarray(sigmas)

    q2_centers, q2_sigma = slice_resolution(logq2[signal], logq2_true[signal], np.arange(-2.0, 0.2 + 1e-9, 0.1))
    pn_centers, pn_sigma = slice_resolution(pn[signal], pn_true[signal], np.arange(0.0, 1.0 + 1e-9, 0.1))
    return (lambda x: np.interp(x, q2_centers, q2_sigma)), (lambda y: np.interp(y, pn_centers, pn_sigma))


def min_width_over_resolution(edges, resolution_at):
    '''Smallest bin-width / local-resolution ratio over a 1D grid, as in notebook 17.'''
    edges = np.asarray(edges, dtype=float)
    centers = 0.5 * (edges[:-1] + edges[1:])
    return float(np.min(np.diff(edges) / resolution_at(centers)))


FAMILY_ROOT = STUDY_ROOT / FAMILY
variant_info = scan_family(FAMILY)
print(f'family {FAMILY!r} at {FAMILY_ROOT} (XML edges from {XML_ROOT / xml_family(FAMILY)})')

VARIANTS = sorted(variant_info, key=lambda name: (np.nan_to_num(variant_info[name]['nbins'], nan=1e9), name))

if RESOLUTION_SAFE_ONLY:
    resolution_at_q2, resolution_at_pn = reco_resolution(RESOLUTION_INPUT_FILE)

    def resolution_safe(name):
        edges = variant_info[name]['edges']
        return edges is not None and (
            min_width_over_resolution(edges[0], resolution_at_q2) >= RESOLUTION_FACTOR
            and min_width_over_resolution(edges[1], resolution_at_pn) >= RESOLUTION_FACTOR)

    unsafe = [v for v in VARIANTS if not resolution_safe(v)]
    if REFERENCE_VARIANT in unsafe:
        raise RuntimeError(
            f'RESOLUTION_SAFE_ONLY: the reference variant {REFERENCE_VARIANT!r} itself has a bin narrower '
            f'than {RESOLUTION_FACTOR:g}x the reco resolution; pick a resolution-safe REFERENCE_VARIANT or turn the flag off')
    if unsafe:
        print(f'RESOLUTION_SAFE_ONLY: dropping {len(unsafe)} variant(s) with a bin narrower than '
              f'{RESOLUTION_FACTOR:g}x resolution: {unsafe}')
        VARIANTS = [v for v in VARIANTS if v not in unsafe]

if REFERENCE_VARIANT not in variant_info:
    raise RuntimeError(f'reference variant {REFERENCE_VARIANT!r} not found below {STUDY_ROOT}; have {VARIANTS}')

candidate_fits = FIT_ORDER if FIT_KEYS is None else list(FIT_KEYS)
FITS = [fit for fit in candidate_fits if any(fit in variant_info[v]['finished'] for v in VARIANTS)]
if not FITS:
    raise RuntimeError('no finished fit found in any variant')

INVENTORY = pd.DataFrame(
    {v: {fit: ('done' if fit in variant_info[v]['finished'] else '') for fit in FITS} for v in VARIANTS}
)
INVENTORY.loc['bins'] = {v: variant_info[v]['nbins'] for v in VARIANTS}
display(INVENTORY)
for v in VARIANTS:
    edges = variant_info[v]['edges']
    if edges:
        print(f'{v:14s} {len(edges[0]) - 1:2d} x {len(edges[1]) - 1:2d}: '
              f'log10Q2 {" ".join(f"{e:g}" for e in edges[0])} | p_n {" ".join(f"{e:g}" for e in edges[1])}')

# One colour per variant, darker for finer binning; the reference is black.
_cmap = mpl.colormaps['viridis']
VARIANT_COLOR = {v: ('black' if v == REFERENCE_VARIANT else _cmap(0.15 + 0.7 * i / max(len(VARIANTS) - 1, 1)))
                 for i, v in enumerate(VARIANTS)}
VARIANT_LABEL = {v: f'{v} ({variant_info[v]["nbins"]:.0f} bins)' for v in VARIANTS}

## 2. Goodness of fit

Same bookkeeping as notebook 3: the pull penalty of Gaussian-constrained parameters is removed
from PROfit's total so that $\chi^2_{\rm data}$ is the spectrum term, and
$N_{\rm dof}=N_{\rm bins}-N_{\rm fitted}$. A drop in $\chi^2_{\rm data}/N_{\rm dof}$ with finer
bins means the finer spectrum is described better *per bin*, which on NuWro fake data mostly
tells you how much model mismatch the coarse bins were averaging over.

For a `_cnp` or `_pearson` family the total is that statistic rather than Neyman's, and it is still labelled chi2 here.


In [ ]:
CHI2_PATTERN = re.compile(r'Global Best Fit chi(?:\^?2|²):\s*([-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?)')
NBINS_PATTERN = re.compile(r'/\s*(\d+)\s+reco bins')
PARAM_PATTERN = re.compile(r'\|\|\s+(\w+)\s+:\s*([-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?)')


def read_fit_stats(log_file):
    '''Total chi2, active-bin count and best-fit parameters from a profile.log.'''
    if not log_file.is_file():
        return np.nan, np.nan, {}
    text = log_file.read_text(errors='replace')
    chi2 = CHI2_PATTERN.findall(text)
    nbins = NBINS_PATTERN.findall(text)
    final_block = text.rsplit('Global Best Fit chi^2:', 1)[-1]
    parameters = {name: float(value) for name, value in PARAM_PATTERN.findall(final_block)}
    return (float(chi2[-1]) if chi2 else np.nan), (int(nbins[-1]) if nbins else np.nan), parameters


records = []
for v in VARIANTS:
    for fit in FITS:
        if fit not in variant_info[v]['finished']:
            continue
        chi2_tot, nbins_log, parameters = read_fit_stats(FAMILY_ROOT / v / fit / 'profile.log')
        penalty = sum(value ** 2 for name, value in parameters.items() if name not in UNIFORM_PARAMETERS.get(fit, set()))
        nbins = nbins_log if np.isfinite(nbins_log) else variant_info[v]['nbins']
        if np.isfinite(nbins_log) and nbins_log != variant_info[v]['nbins']:
            print(f'WARNING {v}/{fit}: the log reports {nbins_log} reco bins but the XML now has '
                  f'{variant_info[v]["nbins"]:.0f}; the result predates the current XML, rerun it')
        ndof = nbins - N_FITTED[fit]
        records.append(dict(variant=v, fit=fit, nbins=nbins, ndof=ndof, chi2_tot=chi2_tot,
                            chi2_penalty=penalty, chi2_data=chi2_tot - penalty,
                            chi2_data_per_ndof=(chi2_tot - penalty) / ndof, **{f'bf_{k}': val for k, val in parameters.items()}))
CHI2 = pd.DataFrame(records)
# Bin count actually fitted, per (variant, fit); falls back to the XML count when a log is missing.
NBINS = {(row.variant, row.fit): row.nbins for row in CHI2.itertuples()}
chi2_table = CHI2.pivot(index='fit', columns='variant', values=['chi2_data', 'chi2_data_per_ndof', 'chi2_penalty'])
chi2_table = chi2_table.reindex(index=FITS).reindex(columns=pd.MultiIndex.from_product(
    [['chi2_data', 'chi2_data_per_ndof', 'chi2_penalty'], VARIANTS]))
display(chi2_table.style.format('{:.3f}', na_rep=''))

## 3. Posterior chains

`load_fit` transforms every chain to physical parameters ($M_A$, or the complete $z$-expansion
coefficient vector). Loading is the slow step; everything below reuses `RESULTS[variant][fit]`.

In [ ]:
RESULTS = {v: {} for v in VARIANTS}
for v in VARIANTS:
    for fit in FITS:
        if fit not in variant_info[v]['finished']:
            continue
        result = load_fit(SPEC[fit], suite=v, burn_in=BURN_IN, thin=THIN, data_root=FAMILY_ROOT)
        if result is None:
            continue
        RESULTS[v][fit] = result
        print(f'{v:14s} {fit:28s} {len(result["samples"]):>8,} samples')

# The production fits were run with MCMC_ITERATIONS=500000, PROfit's default is 20000. Chains
# of different length are still comparable in width and median, but the shorter ones carry more
# MCMC noise, so flag any mismatch against the reference variant.
for fit in FITS:
    lengths = {v: len(RESULTS[v][fit]['samples']) for v in VARIANTS if fit in RESULTS[v]}
    reference_length = lengths.get(REFERENCE_VARIANT)
    if reference_length and any(n != reference_length for n in lengths.values()):
        print(f'WARNING {fit}: chain lengths differ from the reference '
              + ', '.join(f'{v}={n:,}' for v, n in lengths.items())
              + ' -- rerun with MCMC_ITERATIONS set to the same value for a like-for-like comparison')

## 4. Fitted-parameter widths

For each fit the independent physical parameters ($M_A$, or $a_1\ldots a_{k_{\max}-4}$): the
posterior median with its 68% interval, the half-width $(q_{84}-q_{16})/2$, the half-width
relative to the reference variant, and the shift of the median from the reference in units of the
reference half-width.

In [ ]:
def joint_summary(result):
    summary = result['summary'].iloc[result['joint_indices']]
    half_width = 0.5 * (summary['q84'] - summary['q16'])
    return pd.DataFrame({'median': summary['posterior_median'], 'minus': summary['minus_1sigma'],
                         'plus': summary['plus_1sigma'], 'half_width': half_width})


rows = []
for fit in FITS:
    reference = RESULTS[REFERENCE_VARIANT].get(fit)
    ref_summary = joint_summary(reference) if reference is not None else None
    for v in VARIANTS:
        result = RESULTS[v].get(fit)
        if result is None:
            continue
        summary = joint_summary(result)
        for name, row in summary.iterrows():
            ref = ref_summary.loc[name] if ref_summary is not None else None
            rows.append(dict(
                fit=fit, parameter=name, variant=v, median=row['median'], minus=row['minus'], plus=row['plus'],
                half_width=row['half_width'],
                width_ratio=(row['half_width'] / ref['half_width']) if ref is not None else np.nan,
                shift_sigma=((row['median'] - ref['median']) / ref['half_width']) if ref is not None else np.nan,
            ))
PARAMS = pd.DataFrame(rows)
param_table = PARAMS.pivot(index=['fit', 'parameter'], columns='variant', values=['median', 'half_width', 'width_ratio', 'shift_sigma'])
param_table = param_table.reindex(columns=pd.MultiIndex.from_product([['median', 'half_width', 'width_ratio', 'shift_sigma'], VARIANTS]))
display(param_table.style.format('{:.3f}', na_rep=''))

## 5. $F_A(Q^2)$ precision

$\delta_F(Q^2)=\sigma[F_A(Q^2)]/|\mathbb{E}[F_A(Q^2)]|$ from every posterior sample (no subsampling), with the definition of notebook 6, plus the same quantity for the fit's own prior (drawn from `prior_samples`, so for
the uniform-prior fits it reflects the flat sampling range and is not a meaningful reference).
The mean shift is $\mathbb{E}[F_A]_{\rm variant}-\mathbb{E}[F_A]_{\rm ref}$ in units of
$\sigma[F_A]_{\rm ref}$.

In [ ]:
def fa_band(result, q2, use_prior=False, quantiles=(16, 50, 84)):
    '''Mean, standard deviation and percentiles of F_A(Q^2) over EVERY chain sample.

    `_fa_curves` is called with max_samples equal to the chain length so nothing is subsampled;
    the Q^2 points are processed in chunks of Q2_CHUNK so a 500k-sample chain needs ~100 MB at a
    time instead of ~1 GB.
    '''
    q2 = np.atleast_1d(np.asarray(q2, dtype=float))
    n = len(result['prior_samples' if use_prior else 'samples'])
    mean, sigma = np.empty(len(q2)), np.empty(len(q2))
    pct = np.empty((len(quantiles), len(q2)))
    for start in range(0, len(q2), Q2_CHUNK):
        stop = min(start + Q2_CHUNK, len(q2))
        # _fa_curves needs at least two points; pad a single point and drop it again.
        points = q2[start:stop] if stop - start > 1 else np.r_[q2[start:stop], q2[start:stop] + 1e-6]
        curves = _fa_curves(result, points, use_prior=use_prior, max_samples=n)[:, :stop - start]
        mean[start:stop], sigma[start:stop] = curves.mean(axis=0), curves.std(axis=0)
        pct[:, start:stop] = np.percentile(curves, quantiles, axis=0)
    return dict(mean=mean, sigma=sigma, frac=sigma / np.abs(mean), q16=pct[0], median=pct[1], q84=pct[2])


def fa_statistics(result, q2, use_prior=False):
    band = fa_band(result, q2, use_prior=use_prior)
    return band['mean'], band['sigma'], band['frac']


rows = []
for fit in FITS:
    reference = RESULTS[REFERENCE_VARIANT].get(fit)
    ref_mean, ref_sigma, ref_frac = fa_statistics(reference, Q2_REFERENCE) if reference is not None else (None, None, None)
    for v in VARIANTS:
        result = RESULTS[v].get(fit)
        if result is None:
            continue
        mean, sigma, frac = fa_statistics(result, Q2_REFERENCE)
        _, _, prior_frac = fa_statistics(result, Q2_REFERENCE, use_prior=True)
        rows.append(dict(
            fit=fit, variant=v, nbins=NBINS.get((v, fit), variant_info[v]['nbins']),
            **{'F_A mean': mean[0], 'sigma[F_A]': sigma[0], 'posterior [%]': 100 * frac[0], 'prior [%]': 100 * prior_frac[0]},
            **{'ratio to reference': (frac[0] / ref_frac[0]) if reference is not None else np.nan,
               'mean shift [sigma_ref]': ((mean[0] - ref_mean[0]) / ref_sigma[0]) if reference is not None else np.nan},
        ))
FA = pd.DataFrame(rows)
fa_table = FA.pivot(index='fit', columns='variant', values=['posterior [%]', 'ratio to reference', 'mean shift [sigma_ref]'])
fa_table = fa_table.reindex(index=FITS).reindex(columns=pd.MultiIndex.from_product(
    [['posterior [%]', 'ratio to reference', 'mean shift [sigma_ref]'], VARIANTS]))
print(f'delta_F at Q^2 = {Q2_REFERENCE} GeV^2')
display(fa_table.style.format('{:.3f}', na_rep=''))

### Headline: precision and goodness of fit versus number of bins

One line per fit. If finer binning helps, $\delta_F$ falls with the bin count; a flat line means
the extra bins carry no additional information about $F_A$ given the systematics.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
fit_cmap = mpl.colormaps['tab10']
for i, fit in enumerate(FITS):
    fa = FA[FA.fit == fit].sort_values('nbins')
    chi = CHI2[CHI2.fit == fit].sort_values('nbins')
    color = fit_cmap(i % 10)
    axes[0].plot(fa.nbins, fa['posterior [%]'], 'o-', color=color, label=fit)
    axes[1].plot(fa.nbins, fa['ratio to reference'], 'o-', color=color)
    axes[2].plot(chi.nbins, chi.chi2_data_per_ndof, 'o-', color=color)
axes[0].set_ylabel(rf'$\delta_F(Q^2={Q2_REFERENCE})$ [%]')
axes[1].set_ylabel(r'$\delta_F$ / $\delta_F$(reference)'); axes[1].axhline(1, color='grey', lw=0.8)
axes[2].set_ylabel(r'$\chi^2_{\rm data}/N_{\rm dof}$'); axes[2].axhline(1, color='grey', lw=0.8)
for ax in axes:
    ax.set_xlabel('reco bins')
axes[0].legend(fontsize=7, ncol=2)
fig.tight_layout(); plt.show()

### $\delta_F(Q^2)$ and the posterior $F_A$ band per fit

Left: fractional uncertainty versus $Q^2$, one solid line per variant (the reference is black),
the fit's own prior dashed. Right: posterior median and 68% band of $-F_A$ per variant, with the
ratio to the reference median below.

In [ ]:
for fit in FITS:
    available = [v for v in VARIANTS if fit in RESULTS[v]]
    if not available:
        continue
    fig = plt.figure(figsize=(14, 5))
    grid = fig.add_gridspec(2, 2, height_ratios=[3, 1.2], hspace=0.05, wspace=0.25)
    ax_frac = fig.add_subplot(grid[:, 0]); ax_fa = fig.add_subplot(grid[0, 1]); ax_ratio = fig.add_subplot(grid[1, 1], sharex=ax_fa)
    reference = RESULTS[REFERENCE_VARIANT].get(fit)
    ref_median = None
    if reference is not None:
        ref_median = fa_band(reference, Q2)['median']
    for v in available:
        result = RESULTS[v][fit]
        band = fa_band(result, Q2)
        ax_frac.plot(Q2, 100 * band['frac'], color=VARIANT_COLOR[v], lw=1.8 if v == REFERENCE_VARIANT else 1.3, label=VARIANT_LABEL[v])
        q16, med, q84 = band['q16'], band['median'], band['q84']
        ax_fa.plot(Q2, -med, color=VARIANT_COLOR[v], lw=1.3)
        ax_fa.fill_between(Q2, -q84, -q16, color=VARIANT_COLOR[v], alpha=0.15, lw=0)
        if ref_median is not None:
            with np.errstate(divide='ignore', invalid='ignore'):
                ax_ratio.plot(Q2, med / ref_median, color=VARIANT_COLOR[v], lw=1.3)
                ax_ratio.fill_between(Q2, q16 / ref_median, q84 / ref_median, color=VARIANT_COLOR[v], alpha=0.12, lw=0)
    if reference is not None:
        _, _, prior_frac = fa_statistics(reference, Q2, use_prior=True)
        ax_frac.plot(Q2, 100 * prior_frac, color='grey', ls='--', lw=1.2, label='prior')
    ax_frac.axvline(Q2_REFERENCE, color='grey', lw=0.8, ls=':')
    ax_frac.set_xlabel(r'$Q^2$ [GeV$^2$]'); ax_frac.set_ylabel(r'$\delta_F(Q^2)$ [%]'); ax_frac.legend(fontsize=8)
    ax_frac.set_ylim(0, min(ax_frac.get_ylim()[1], 60))
    ax_fa.set_ylabel(r'$-F_A(Q^2)$'); ax_fa.tick_params(labelbottom=False)
    ax_ratio.axhline(1, color='grey', lw=0.8); ax_ratio.set_ylabel('median / ref.'); ax_ratio.set_xlabel(r'$Q^2$ [GeV$^2$]')
    fig.suptitle(f'{fit}: {SPEC[fit].title}', y=0.98)
    plt.show()

## 6. Axial radius $r_A^2$

The slope of $F_A$ at $Q^2=0$, with the definitions of notebook 7: $r_A^2 = 12/M_A^2$ for the
dipole fits and $r_A^2=-\frac{6}{g_A}\,\frac{dF_A}{dQ^2}\big|_0$ evaluated analytically from the
$z$-expansion coefficients otherwise, in fm$^2$. The table gives the median with its 68%
interval, the half-width relative to the reference, and the fraction of posterior samples with
$r_A^2<0$. That fraction is the number to watch: a binning that resolves the low-$Q^2$ slope
should drive it towards zero. The fit's own prior is shown dashed in the figures for scale.

In [ ]:
HBARC_GEV_FM = 0.1973269804


def radius_squared_from_ma(ma_gev):
    return 12.0 / np.asarray(ma_gev, dtype=float) ** 2 * HBARC_GEV_FM ** 2


def dz_dq2_at_zero(t0_gev2, t_cut_gev2):
    a, b = np.sqrt(t_cut_gev2), np.sqrt(t_cut_gev2 - t0_gev2)
    return b / (a * (a + b) ** 2)


def radius_squared_from_zexp(coefficients, t0_gev2, t_cut_gev2, g_a):
    coefficients = np.atleast_2d(np.asarray(coefficients, dtype=float))
    z0 = ((np.sqrt(t_cut_gev2) - np.sqrt(t_cut_gev2 - t0_gev2)) /
          (np.sqrt(t_cut_gev2) + np.sqrt(t_cut_gev2 - t0_gev2)))
    k = np.arange(1, coefficients.shape[1])
    dfa_dq2 = (coefficients[:, 1:] @ (k * z0 ** (k - 1))) * dz_dq2_at_zero(t0_gev2, t_cut_gev2)
    return -6.0 / g_a * dfa_dq2 * HBARC_GEV_FM ** 2


def radius_squared(result, use_prior=False):
    samples = result['prior_samples' if use_prior else 'samples']
    prior = result['spec'].prior
    if prior is None:
        return radius_squared_from_ma(samples[:, 0])
    return radius_squared_from_zexp(samples, prior.t0_gev2, prior.t_cut_gev2, prior.fa_q2_zero)


R2 = {v: {fit: radius_squared(RESULTS[v][fit]) for fit in RESULTS[v]} for v in VARIANTS}
rows = []
for fit in FITS:
    ref = R2[REFERENCE_VARIANT].get(fit)
    ref_q16, ref_med, ref_q84 = np.percentile(ref, [16, 50, 84]) if ref is not None else (np.nan,) * 3
    for v in VARIANTS:
        if fit not in R2[v]:
            continue
        r2 = R2[v][fit]
        q16, med, q84 = np.percentile(r2, [16, 50, 84])
        rows.append(dict(
            fit=fit, variant=v, nbins=NBINS.get((v, fit), variant_info[v]['nbins']),
            **{'median [fm^2]': med, 'minus': med - q16, 'plus': q84 - med, 'half_width': 0.5 * (q84 - q16),
               'width_ratio': 0.5 * (q84 - q16) / (0.5 * (ref_q84 - ref_q16)) if ref is not None else np.nan,
               'shift_sigma': (med - ref_med) / (0.5 * (ref_q84 - ref_q16)) if ref is not None else np.nan,
               'negative [%]': 100 * np.mean(r2 < 0)},
        ))
RADIUS = pd.DataFrame(rows)
radius_table = RADIUS.pivot(index='fit', columns='variant', values=['median [fm^2]', 'half_width', 'width_ratio', 'negative [%]'])
radius_table = radius_table.reindex(index=FITS).reindex(columns=pd.MultiIndex.from_product(
    [['median [fm^2]', 'half_width', 'width_ratio', 'negative [%]'], VARIANTS]))
display(radius_table.style.format('{:.3f}', na_rep=''))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for i, fit in enumerate(FITS):
    part = RADIUS[RADIUS.fit == fit].sort_values('nbins')
    color = fit_cmap(i % 10)
    axes[0].plot(part.nbins, part['negative [%]'], 'o-', color=color, label=fit)
    axes[1].plot(part.nbins, part['width_ratio'], 'o-', color=color)
    axes[2].errorbar(part.nbins + 1.5 * i, part['median [fm^2]'], yerr=[part['minus'], part['plus']], fmt='o-', ms=3, color=color, lw=1, capsize=2)
axes[0].set_ylabel(r'samples with $r_A^2<0$ [%]')
axes[1].set_ylabel(r'$r_A^2$ half-width / reference'); axes[1].axhline(1, color='grey', lw=0.8)
axes[2].set_ylabel(r'$r_A^2$ median and 68% interval [fm$^2$]'); axes[2].axhline(0, color='grey', lw=0.8)
for ax in axes:
    ax.set_xlabel('reco bins')
axes[0].legend(fontsize=7, ncol=2)
fig.tight_layout(); plt.show()

# Posterior r_A^2 per variant, one panel per fit; the fit's own prior dashed.
ncol = 3
nrow = int(np.ceil(len(FITS) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(5 * ncol, 3.3 * nrow), squeeze=False)
for ax, fit in zip(axes.flat, FITS):
    available = [v for v in VARIANTS if fit in R2[v]]
    allvals = np.concatenate([R2[v][fit] for v in available])
    lo, hi = np.percentile(allvals, [0.5, 99.5])
    lo, hi = min(lo, -0.05), max(hi, 0.05)
    bins = np.linspace(lo, hi, 80)
    for v in available:
        ax.hist(R2[v][fit], bins=bins, density=True, histtype='step', color=VARIANT_COLOR[v],
                lw=1.6 if v == REFERENCE_VARIANT else 1.1, label=VARIANT_LABEL[v])
    reference = RESULTS[REFERENCE_VARIANT].get(fit) or RESULTS[available[0]][fit]
    ax.hist(radius_squared(reference, use_prior=True), bins=bins, density=True, histtype='step', color='grey', ls='--', lw=1, label='prior')
    ax.axvline(0, color='red', lw=0.8)
    ax.set_title(fit, fontsize=9); ax.set_xlabel(r'$r_A^2$ [fm$^2$]'); ax.set_yticks([])
for ax in axes.flat[len(FITS):]:
    ax.axis('off')
axes.flat[0].legend(fontsize=7)
fig.tight_layout(); plt.show()

## 7. Profile scans of one fit

PROfit's 1D profile scans, $\Delta\chi^2$ against each fitted parameter (in fit coordinates:
pulls for the $z$-expansion PCA directions, knob units for $M_A$), overlaid across variants for
`PROFILE_SCAN_FIT`. Narrower parabolas mean a stronger constraint; a displaced minimum means the
finer binning prefers a different best fit.

In [ ]:
PROFILE_SCAN_FIT = 'ma_no_axff'

def read_profile_points(variant, fit):
    matches = sorted((FAMILY_ROOT / variant / fit).glob('*_v1_PROfile_points.txt'))
    if not matches:
        return None
    rows = []
    for line in matches[0].read_text().splitlines():
        if not line or line.startswith('#'):
            continue
        index, kind, name, value, dchi2, chi2 = line.split()
        rows.append(dict(index=int(index), kind=kind, parameter=name, value=float(value), dchi2=float(dchi2), chi2=float(chi2)))
    return pd.DataFrame(rows)


scans = {v: read_profile_points(v, PROFILE_SCAN_FIT) for v in VARIANTS if PROFILE_SCAN_FIT in variant_info[v]['finished']}
scans = {v: s for v, s in scans.items() if s is not None and not s.empty}
if not scans:
    print(f'no profile scans found for {PROFILE_SCAN_FIT!r}')
else:
    parameters = list(dict.fromkeys(p for s in scans.values() for p in s.parameter))
    fig, axes = plt.subplots(1, len(parameters), figsize=(4.5 * len(parameters), 3.8), squeeze=False)
    for ax, parameter in zip(axes[0], parameters):
        for v, scan in scans.items():
            part = scan[scan.parameter == parameter].sort_values('value')
            ax.plot(part.value, part.dchi2, 'o-', ms=3, color=VARIANT_COLOR[v], label=VARIANT_LABEL[v])
        ax.axhline(1, color='grey', lw=0.8, ls=':'); ax.axhline(4, color='grey', lw=0.8, ls=':')
        ax.set_xlabel(parameter); ax.set_ylabel(r'$\Delta\chi^2$'); ax.set_ylim(0, min(ax.get_ylim()[1], 30))
    axes[0][0].legend(fontsize=8)
    fig.suptitle(f'{PROFILE_SCAN_FIT}: profile scans per variant', y=1.02)
    fig.tight_layout(); plt.show()

## 8. Cross-family overlay

The same headline quantities for several families, for the fits in `COMPARE_FITS`: one panel
per fit, one line per family, against the number of bins. Read it as

* **asimov**: drift here is an estimator artifact, since the fake data is the prediction itself;
* **nuwro_pm3 vs nuwro**: the effect of the +-3 spline fit box that clipped PCA2 in the first batch;
* **nuwro vs nuwro_cnp**: the Neyman-versus-CNP dependence, i.e. how much of the shift the
  low-count bins are driving through the statistic.

Families that do not exist yet are skipped. Every chain of every listed family is loaded, so
this section takes a few minutes per family.

In [ ]:
FAMILY_STYLE = {'nuwro_pm3': ':', 'nuwro': '-', 'nuwro_cnp': '--', 'asimov': '-.'}
compare_rows = []
for family in COMPARE_FAMILIES:
    if not (STUDY_ROOT / family).is_dir():
        print(f'{family:12s} not run yet, skipped')
        continue
    info = scan_family(family)
    for v in sorted(info, key=lambda name: (np.nan_to_num(info[name]['nbins'], nan=1e9), name)):
        for fit in COMPARE_FITS:
            if fit not in info[v]['finished']:
                continue
            chi2_tot, nbins_log, parameters = read_fit_stats(STUDY_ROOT / family / v / fit / 'profile.log')
            penalty = sum(val ** 2 for name, val in parameters.items() if name not in UNIFORM_PARAMETERS.get(fit, set()))
            nbins = nbins_log if np.isfinite(nbins_log) else info[v]['nbins']
            result = RESULTS[v][fit] if (family == FAMILY and fit in RESULTS.get(v, {})) else load_fit(
                SPEC[fit], suite=v, burn_in=BURN_IN, thin=THIN, data_root=STUDY_ROOT / family)
            if result is None:
                continue
            mean, sigma, frac = fa_statistics(result, Q2_REFERENCE)
            r2 = radius_squared(result)
            joint = joint_summary(result)
            compare_rows.append(dict(
                family=family, variant=v, fit=fit, nbins=nbins,
                chi2_per_ndof=(chi2_tot - penalty) / (nbins - N_FITTED[fit]),
                deltaF=100 * frac[0], fa_mean=mean[0], fa_sigma=sigma[0],
                r2_median=np.median(r2), r2_negative=100 * np.mean(r2 < 0),
                first_param=joint.index[0], first_median=joint['median'].iloc[0], first_half_width=joint['half_width'].iloc[0],
            ))
COMPARE = pd.DataFrame(compare_rows)
if COMPARE.empty:
    print('nothing to compare yet')
else:
    display(COMPARE.pivot_table(index=['fit', 'variant'], columns='family',
                                values=['chi2_per_ndof', 'deltaF', 'r2_median', 'r2_negative'], sort=False)
            .style.format('{:.3f}', na_rep=''))

In [ ]:
if not COMPARE.empty:
    fits = [fit for fit in COMPARE_FITS if fit in set(COMPARE.fit)]
    quantities = [('chi2_per_ndof', r'$\chi^2_{\rm data}/N_{\rm dof}$'), ('deltaF', rf'$\delta_F(Q^2={Q2_REFERENCE})$ [%]'),
                  ('fa_mean', rf'$\mathbb{{E}}[F_A]$ at $Q^2={Q2_REFERENCE}$'), ('r2_median', r'$r_A^2$ median [fm$^2$]'),
                  ('r2_negative', r'$r_A^2<0$ [%]')]
    fig, axes = plt.subplots(len(quantities), len(fits), figsize=(4.2 * len(fits), 2.9 * len(quantities)), squeeze=False, sharex='col')
    for j, fit in enumerate(fits):
        for i, (column, label) in enumerate(quantities):
            ax = axes[i][j]
            for family in COMPARE_FAMILIES:
                part = COMPARE[(COMPARE.fit == fit) & (COMPARE.family == family)].sort_values('nbins')
                if part.empty:
                    continue
                ax.plot(part.nbins, part[column], 'o' + FAMILY_STYLE.get(family, '-'), ms=4, label=family)
            if column == 'chi2_per_ndof':
                ax.axhline(1, color='grey', lw=0.8)
            if column in ('r2_median',):
                ax.axhline(0, color='grey', lw=0.8)
            if j == 0:
                ax.set_ylabel(label)
            if i == 0:
                ax.set_title(fit)
            if i == len(quantities) - 1:
                ax.set_xlabel('reco bins')
    axes[0][0].legend(fontsize=8)
    fig.tight_layout(); plt.show()

    # Same overlay for the first fitted physical parameter (M_A or a_1) with its 68% half-width.
    fig, axes = plt.subplots(1, len(fits), figsize=(4.2 * len(fits), 3.4), squeeze=False)
    for ax, fit in zip(axes[0], fits):
        for k, family in enumerate(COMPARE_FAMILIES):
            part = COMPARE[(COMPARE.fit == fit) & (COMPARE.family == family)].sort_values('nbins')
            if part.empty:
                continue
            ax.errorbar(part.nbins + 2 * k, part.first_median, yerr=part.first_half_width, fmt='o' + FAMILY_STYLE.get(family, '-'), ms=4, capsize=2, label=family)
        ax.set_title(fit); ax.set_xlabel('reco bins'); ax.set_ylabel(part.first_param.iloc[0] if not part.empty else '')
    axes[0][0].legend(fontsize=8)
    fig.tight_layout(); plt.show()